# 畳み込みニューラルネットワーク（CNN）Functional API編

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARIM-ACADEMY-2026/Advanced_Tutorial_3_Keras/blob/main/2_Keras_CNN_Functional-API.ipynb)

`2_Keras_CNN_Sequential-API.ipynb`と**全く同じモデル構成・同じデータセット（Fashion-MNIST）**を、書き方（API）だけを変えて実装します。同じ結果を2通りの書き方で再現することで、Sequential APIとFunctional APIの違いを体感するのがこのノートブックの目的です。

## 対象読者・前提知識・動作環境・版とライセンス

- **対象読者**: `2_Keras_CNN_Sequential-API.ipynb`を読んだ方（同じCNNをFunctional APIで書き直す内容のため）
- **前提知識**: `Conv2D`・`MaxPooling2D`など畳み込み層の基本はSequential編で説明済みとして進める
- **動作環境**: Python 3.10以降、TensorFlow 2.16以降・Keras 3系（`tensorflow.keras`は現在Keras 3を指す）
- **データセット**: Fashion-MNIST（Zalando Research作成、MITライセンス）
- **版**: 2026-08-03作成

## 目次

1. データの準備（Sequential編と共通）
2. モデルの構築（Functional API）
3. 学習
4. 評価・API書式の比較とまとめ

## 教材への接続（Google Colab）

Google Colabで開いた場合は、次のセルを実行してこのリポジトリをクローンし、`module/`（共通ヘルパー関数）や`output/`・`comparison_output/`（他ノートブックとの受け渡しファイル）を含むフォルダへ移動してください。ローカル環境でこのフォルダを直接開いている場合は、次のセルは実行不要です（すでにカレントディレクトリが正しい場所になっています）。


In [ ]:
!git clone https://github.com/ARIM-ACADEMY-2026/Advanced_Tutorial_3_Keras.git
%cd Advanced_Tutorial_3_Keras


## 1. データの準備

### ステップ1: ライブラリを読み込み、乱数シードを固定する

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib_fontja

import warnings
warnings.filterwarnings('ignore')

In [ ]:
BASE_DIR = Path.cwd()
sys.path.append(str(BASE_DIR))  # module/ はBASE_DIR直下にあるため、BASE_DIR自体をsys.pathに加える

from module.seed_utils import set_seed
from module import data_utils, viz_utils

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

set_seed(42)

### ステップ2: データセットを読み込む（Sequential編と同じ前処理）

In [ ]:
data = data_utils.load_fashion_mnist(add_channel=True, one_hot=True)

train_images, test_images = data.train_images, data.test_images
train_labels, test_labels = data.train_labels, data.test_labels
train_labels_int, test_labels_int = data.train_labels_int, data.test_labels_int
class_names = data.class_names

train_images.shape, test_images.shape

データの前処理はSequential編と完全に同じです（同じ`data_utils.load_fashion_mnist()`を使っています）。違いが出るのはこの先、モデルの組み立て方だけです。

## 2. モデルの構築

### ステップ1: Functional APIとSequential APIの書き方の違い

Sequential APIでは`model.add(層)`のように、層を1つずつモデルへ「追加」していきました。Functional APIでは、各層を**関数のように呼び出し**、前の層の出力を次の層の入力として明示的につないでいきます。

```python
# Sequential API（前作のCNN Sequential編）
model = keras.Sequential()
model.add(Conv2D(64, (3, 3), ...))
model.add(...)

# Functional API（本ノートブック）
inputs = Input(shape=(28, 28, 1))
c1 = Conv2D(64, (3, 3), ...)(inputs)   # 層を「関数」として呼び出し、入力を渡す
...
model = Model(inputs=inputs, outputs=outputs)  # 入口と出口を指定してモデル化
```

一本道のモデルではFunctional APIの利点はあまり感じられませんが、入力が2つある・出力が2つある・層の出力を後方の複数の層で使い回す、といった分岐・合流のあるモデル（このシリーズ後半のVAEなど）ではFunctional APIでないと書けません。「まずは一本道のCNNで書き方に慣れる」というのが本ノートブックの位置づけです。

In [ ]:
from tensorflow import keras
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

inputs = Input(shape=(28, 28, 1))
inputs

`Input(shape=(28, 28, 1))`は、Sequential APIの`input_shape=(28, 28, 1)`に相当する「モデルの入り口」です。この時点ではまだ実データを持たない、形だけを表す特別なテンソル（`KerasTensor`）です。

### ステップ2: 層を1つずつ関数として呼び出す

In [ ]:
c1 = Conv2D(filters=64, kernel_size=(3, 3), padding="same", activation="relu")(inputs)
s2 = MaxPooling2D(pool_size=(2, 2))(c1)
c1.shape, s2.shape

`Conv2D(...)(inputs)`という書き方に注目してください。`Conv2D(...)`の部分で層のインスタンス（設定）を作り、続けて`(inputs)`とすることでその層に`inputs`を「通す」処理をしています。層のインスタンスを作る行程と、実際にデータを流す行程が分離されているのがFunctional APIの特徴です。

In [ ]:
c3 = Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation="relu")(s2)
s4 = MaxPooling2D(pool_size=(2, 2))(c3)
s4.shape

In [ ]:
f = Flatten()(s4)
c5 = Dense(units=128, activation="relu")(f)
f6 = Dense(units=64, activation="relu")(c5)
outputs = Dense(units=10, activation="softmax")(f6)
outputs.shape

### ステップ3: 入口と出口を指定してモデルを作る

In [ ]:
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

`keras.Model(inputs=..., outputs=...)`で、「どこが入口で、どこが出口か」を指定してはじめて1つのモデルになります。`model.summary()`の内容（各層の出力形状・パラメータ数）は、Sequential編で作ったモデルと完全に一致するはずです。**書き方が違うだけで、できあがるモデルは同じもの**という点を確認してください（4節で実際に検証します）。

**コラム: 中間変数`c1`, `s2`, `c3`, `s4`, ...に短い名前を使う理由**

Functional APIでは各層の出力をいったん変数に受けるため、変数名を考える必要があります。ここでは元の教材にならい`c1`（畳み込み1）、`s2`（サブサンプリング＝プーリング2）のように「層の種類の頭文字＋通し番号」で命名しています。実務では`conv1_output`のように意味のわかる名前を使うことも多いですが、層の数が多いモデルでは短い記号的な名前の方が見通しがよい場合もあります。プロジェクト・チームの流儀に合わせて選んでください。

### ステップ4: モデルをコンパイルする

In [ ]:
model.compile(
    optimizer="rmsprop",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

## 3. 学習

In [ ]:
%%time
history = model.fit(
    train_images,
    train_labels,
    epochs=30,
    batch_size=128,
    validation_data=(test_images, test_labels),
)

## 4. 評価・API書式の比較とまとめ

### ステップ1: Sequential編と同じ構造になっているか確認する

In [ ]:
seq_model_path = BASE_DIR / "output" / "cnn_seq_trained.keras"

if seq_model_path.exists():
    seq_model = keras.models.load_model(seq_model_path)
    same_param_count = model.count_params() == seq_model.count_params()
    same_shapes = [l.output_shape for l in model.layers if hasattr(l, "output_shape")] == \
                  [l.output_shape for l in seq_model.layers if hasattr(l, "output_shape")]
    print("パラメータ数が一致:", same_param_count,
          f"(Functional: {model.count_params()}, Sequential: {seq_model.count_params()})")
    assert same_param_count, "SequentialとFunctionalでパラメータ数が異なる（層の構成を見直す）"
else:
    print("Sequential編の output/cnn_seq_trained.keras が見つからないため、比較をスキップしました。"
          " 2_Keras_CNN_Sequential-API.ipynb を先に実行してください。")

`assert`文は「この条件が成り立たなければエラーを出して止める」という書き方です。ここでは「FunctionalとSequentialで書き方は違っても、できあがるモデルの構造（パラメータ数）は一致するはずだ」という主張を、コード自身にチェックさせています。目視での確認に加えて、こうした自己検証をコードに埋め込んでおくと、後で層の構成を変更したときに不整合へすぐ気づけます。

### ステップ2: 損失関数の値と正解率の最終値を確認する

In [ ]:
train_loss_cnn_func, train_acc_cnn_func = model.evaluate(train_images, train_labels, verbose=0)
test_loss_cnn_func, test_acc_cnn_func = model.evaluate(test_images, test_labels, verbose=0)

print("訓練データの損失関数の値:", train_loss_cnn_func)
print("テストデータの損失関数の値:", test_loss_cnn_func)
print("訓練データの分類精度:", train_acc_cnn_func)
print("テストデータの分類精度:", test_acc_cnn_func)

### ステップ3: 学習曲線・混同行列を確認する（図1〜図3）

In [ ]:
viz_utils.plot_metric_curve(history, metric="loss", fig_num=1, output_dir=OUTPUT_DIR)

In [ ]:
viz_utils.plot_metric_curve(history, metric="accuracy", fig_num=2, output_dir=OUTPUT_DIR)

In [ ]:
test_pred_proba_cnn_func = model.predict(test_images, verbose=0)
test_pred_class_cnn_func = np.argmax(test_pred_proba_cnn_func, axis=1)

cm_cnn_func = viz_utils.plot_confusion_matrix(
    test_labels_int, test_pred_class_cnn_func,
    class_names=class_names, fig_num=3, output_dir=OUTPUT_DIR,
)

### ステップ4: 学習済みモデルを保存する

In [ ]:
model_path = OUTPUT_DIR / "cnn_func_trained.keras"
history_path = data_utils.save_history(OUTPUT_DIR, "history_cnn_func.npz", history)
model.save(model_path)
print("保存しました:", model_path, "/", history_path)

### ステップ5: MLP・CNN(Sequential)との比較表・比較図（図4）

In [ ]:
import csv

COMPARISON_DIR = BASE_DIR.parent / "comparison_output"
COMPARISON_DIR.mkdir(exist_ok=True)
comparison_path = COMPARISON_DIR / "classification_comparison.csv"

row = {
    "model": "CNN",
    "api_style": "Functional",
    "dataset": "Fashion-MNIST",
    "test_loss": test_loss_cnn_func,
    "test_acc": test_acc_cnn_func,
}
file_exists = comparison_path.exists()
with open(comparison_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(row.keys()))
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"記録先: {comparison_path}")

In [ ]:
import pandas as pd

comparison_df = pd.read_csv(comparison_path)
comparison_df

In [ ]:
plt.figure(figsize=(9, 5))
labels_for_bar = comparison_df["model"] + " (" + comparison_df["api_style"] + ")"
plt.bar(labels_for_bar, comparison_df["test_acc"], color="steelblue")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)
plt.title("図4: モデル別テスト正解率の比較（Fashion-MNIST）")
for i, acc in enumerate(comparison_df["test_acc"]):
    plt.text(i, acc + 0.02, f"{acc:.3f}", ha="center")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig4_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

`CNN (Sequential)`と`CNN (Functional)`の行は、同じ構造のモデルを同じデータ・同じ乱数シードで学習させたものなので、実行環境が同じであればほぼ同じ正解率になるはずです（浮動小数点演算の順序の違いなどでごくわずかに異なる場合はあります）。書き方（API）を変えても学習結果そのものは変わらない、ということを実際の数値で確認できます。

## まとめ

- Sequential編と同じCNN構成を、Functional APIで書き直した。`層(...)＋(入力)`という関数呼び出しの形で層をつなぎ、最後に`Model(inputs=..., outputs=...)`でモデル化する書き方を学んだ
- パラメータ数の一致を`assert`で検証し、書き方が違っても同一のモデルができることを確認した
- MLP・CNN(Sequential)・CNN(Functional)の3モデルを比較表・比較図にまとめた

## 本ノートブックで扱っていないこと（今後の課題）

- Functional APIならではの利点（分岐・合流のあるモデル）は、次の`2_Keras_VAE_Functional-API.ipynb`で実際に体験する
- モデルのサブクラス化（Subclassing API）という3つ目の書き方（VAE編のカスタム`train_step`で一部触れる）

## 演習問題

1. `keras.utils.plot_model(model, show_shapes=True)`を使ってモデルの構造を図として出力し、`model.summary()`のテキスト表示と見比べなさい（`pydot`と`graphviz`のインストールが必要）。
2. 本ノートブックの`c1`, `s2`, `c3`, `s4`のような短い変数名を、`conv1_out`のような説明的な名前に置き換えて書き直しなさい。可読性はどう変わるか、自分の意見をまとめなさい。
3. Functional APIの利点である「層の出力を複数の場所で使い回す」例として、`c1`（1つ目の畳み込み層の出力）を`Flatten`してから、`f6`と結合（`keras.layers.Concatenate`）した上で出力層に渡すモデルを作ってみなさい（Sequential APIでは書けないことを確認する）。